## Bước 1 – Import thư viện và cấu hình

In [2]:
from pathlib import Path
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

PROJECT_DIR = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent

print("Project:", PROJECT_DIR)

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

# Notebook này yêu cầu GPU NVIDIA có CUDA.
if not torch.cuda.is_available():
    raise RuntimeError("Không tìm thấy GPU CUDA.")

torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device("cuda")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Project: D:\Plant_disease\plant-disease-classification
Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## Bước 2 – Nạp dữ liệu và xử lý ảnh

In [ ]:
DATA_DIR = PROJECT_DIR.parent / "archive" / "PlantVillage"
PREPROCESSING_PATH = PROJECT_DIR / "data" / "preprocessing_data.csv"

processed_df = pd.read_csv(PREPROCESSING_PATH)

class_table = processed_df[["class_id", "class_name"]].drop_duplicates()
class_table = class_table.sort_values("class_id")
class_names = class_table["class_name"].tolist()
num_classes = len(class_names)

IMAGE_SIZE = 224
# Batch 64 đã được kiểm tra trên RTX 3050 6GB.
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001

print("Data directory:", DATA_DIR)
print("Number of classes:", num_classes)
print("Preprocessing data:", PREPROCESSING_PATH)
print(processed_df["split"].value_counts())

class PlantDiseaseDataset(Dataset):
    def __init__(self, dataframe, data_dir, transform):
        self.paths = dataframe["relative_path"].astype(str).tolist()
        self.labels = dataframe["class_id"].astype(int).tolist()
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        relative_path = self.paths[index].replace("\\", "/")
        image_path = self.data_dir.joinpath(*relative_path.split("/"))
        label = self.labels[index]

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        return image, label

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_df = processed_df[processed_df["split"] == "train"]
val_df = processed_df[processed_df["split"] == "validation"]
test_df = processed_df[processed_df["split"] == "test"]

train_dataset = PlantDiseaseDataset(train_df, DATA_DIR, image_transform)
val_dataset = PlantDiseaseDataset(val_df, DATA_DIR, image_transform)
test_dataset = PlantDiseaseDataset(test_df, DATA_DIR, image_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

gpu_train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, scale=(0.90, 1.10)),
    transforms.ColorJitter(contrast=0.10),
])

Train batches: 611
Validation batches: 68
Test batches: 170


## Bước 3 – Xây dựng Complex CNN

In [5]:
class ParallelBlock(nn.Module):
    def __init__(self, input_channels):
        super().__init__()

        # Nhánh 1: convolution 1x1.
        self.branch1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # Nhánh 2: convolution 3x3.
        self.branch2 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # Nhánh 3: convolution 5x5.
        self.branch3 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

    def forward(self, x):
        output1 = self.branch1(x)
        output2 = self.branch2(x)
        output3 = self.branch3(x)

        # 3 nhánh x 32 kênh = 96 kênh.
        output = torch.cat(
            [output1, output2, output3],
            dim=1,
        )

        return output

class ComplexCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.parallel_block = ParallelBlock(input_channels=64)

        self.block3 = nn.Sequential(
            nn.Conv2d(96, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.average_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.5)
        self.output_layer = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.parallel_block(x)
        x = self.block3(x)

        x = self.average_pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = self.output_layer(x)

        return x

model = ComplexCNN(num_classes=num_classes).to(device)
total_parameters = sum(parameter.numel() for parameter in model.parameters())

sample_images = torch.zeros(
    2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device
)

model.eval()
with torch.inference_mode():
    sample_outputs = model(sample_images)

print(model)
print("Model device:", next(model.parameters()).device)
print("Output shape:", sample_outputs.shape)
print(f"Total parameters: {total_parameters:,}")

ComplexCNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (parallel_block): ParallelBlock(
    (branch1): Sequential(
      (0): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (branch2): Sequential(
      (0): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True

## Bước 4 – Huấn luyện

In [6]:
loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

# Mixed precision giúp GPU RTX train nhanh hơn và dùng ít VRAM hơn.
scaler = torch.amp.GradScaler("cuda")

def train_one_epoch(model, dataloader):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in dataloader:
        # Chuyển batch từ RAM lên GPU.
        images = images.to(device)
        labels = labels.to(device)

        images = gpu_train_augmentation(images)

        optimizer.zero_grad()

        # Forward và loss dùng mixed precision trên GPU.
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images)
            loss = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = total_correct / total_images

    return average_loss, accuracy

@torch.inference_mode()
def evaluate(model, dataloader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images)
            loss = loss_function(outputs, labels)
        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = total_correct / total_images

    return average_loss, accuracy

In [ ]:
best_val_loss = float("inf")
best_epoch = 0
best_model_state = None

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = train_one_epoch(model, train_loader)

    val_loss, val_accuracy = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    # Giữ trọng số tốt nhất trong RAM.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_accuracy:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.4f}"
    )

print("Best epoch:", best_epoch)

model.load_state_dict(best_model_state)

# Lưu đúng một checkpoint sau khi đã nạp lại trọng số tốt nhất.
models_dir = PROJECT_DIR / "outputs" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = models_dir / "complex_cnn.pt"
torch.save({
    "format_version": 1,
    "model_name": "complex",
    "state_dict": {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    },
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "normalization": None,
}, checkpoint_path)
print("Đã lưu checkpoint:", checkpoint_path)

Epoch [1/10] | Train Loss: 1.7052 | Train Acc: 0.5382 | Val Loss: 1.1938 | Val Acc: 0.6252
Epoch [2/10] | Train Loss: 1.0406 | Train Acc: 0.6957 | Val Loss: 0.6747 | Val Acc: 0.7958
Epoch [3/10] | Train Loss: 0.8209 | Train Acc: 0.7550 | Val Loss: 0.7014 | Val Acc: 0.7751
Epoch [4/10] | Train Loss: 0.6868 | Train Acc: 0.7930 | Val Loss: 1.1337 | Val Acc: 0.6752
Epoch [5/10] | Train Loss: 0.5951 | Train Acc: 0.8179 | Val Loss: 0.3887 | Val Acc: 0.8833
Epoch [6/10] | Train Loss: 0.5283 | Train Acc: 0.8379 | Val Loss: 0.6149 | Val Acc: 0.8142
Epoch [7/10] | Train Loss: 0.4784 | Train Acc: 0.8511 | Val Loss: 0.3732 | Val Acc: 0.8808
Epoch [8/10] | Train Loss: 0.4391 | Train Acc: 0.8640 | Val Loss: 0.3095 | Val Acc: 0.9056
Epoch [9/10] | Train Loss: 0.3990 | Train Acc: 0.8751 | Val Loss: 0.5693 | Val Acc: 0.8294
Epoch [10/10] | Train Loss: 0.3781 | Train Acc: 0.8820 | Val Loss: 0.9055 | Val Acc: 0.7558
Best epoch: 8


## Bước 5 – Đánh giá trên tập test

In [ ]:
test_loss, test_accuracy = evaluate(model, test_loader)

all_predictions = []
all_labels = []
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.tolist())

precision, recall, f1_score, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0,
)

metrics = {
    "best_epoch": best_epoch,
    "test_loss": test_loss,
    "accuracy": test_accuracy,
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1_score),
}

print(f"Test Loss : {test_loss:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1_score:.4f}")

Test Loss : 0.3127
Accuracy  : 0.9019
Precision : 0.9141
Recall    : 0.9019
F1-score  : 0.9020


In [11]:
# Tính các chỉ số riêng cho từng lớp theo đúng thứ tự class_id.
class_precision, class_recall, class_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        labels=np.arange(num_classes),
        average=None,
        zero_division=0,
    )
)


def readable_class_name(class_name):
    """Đổi Apple___Apple_scab thành Apple / Apple scab."""
    parts = class_name.split("___", maxsplit=1)
    return " / ".join(
        part.replace("_", " ").strip() for part in parts
    )


classification_table = pd.DataFrame({
    "Class": [readable_class_name(name) for name in class_names],
    "Precision": class_precision,
    "Recall": class_recall,
    "F1-score": class_f1,
})

# Định dạng giống bảng báo cáo: ẩn index và hiển thị 4 chữ số thập phân.
styled_classification_table = (
    classification_table.style
    .format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
    })
    .hide(axis="index")
    .set_properties(subset=["Class"], **{"text-align": "left"})
    .set_table_styles([
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        },
        {
            "selector": "th.col_heading:first-child",
            "props": [("text-align", "left")],
        },
    ])
)

styled_classification_table

Class,Precision,Recall,F1-score
Apple / Apple scab,0.6500,0.8254,0.7273
Apple / Black rot,0.6932,0.9760,0.8106
Apple / Cedar apple rust,1.0000,0.7636,0.8660
Apple / healthy,0.8702,0.9722,0.9184
Blueberry / healthy,0.9365,0.9833,0.9593
Cherry (including sour) / Powdery mildew,0.9461,0.9190,0.9324
Cherry (including sour) / healthy,0.8804,0.9529,0.9153
Corn (maize) / Cercospora leaf spot Gray leaf spot,0.8041,0.7573,0.7800
Corn (maize) / Common rust,0.9755,1.0000,0.9876
Corn (maize) / Northern Leaf Blight,0.8108,0.9137,0.8592
